# 14 — Create the App Log Observability Genie Agent

**Jackson and Jackson HR Digital**

Creates (or updates) a Genie Agent over the app log views from notebook `13`, so app owners can ask
plain-English questions about the Hire Right Databricks App's runtime logs — no raw OTel JSON or SQL required.

### What this notebook does
1. Loads the agent definition from `14_app_log_genie_space.json` (instructions that explain the raw
   `attributes` / `app.log_source` fields, column descriptions, synonyms, joins, example SQL, measures, and benchmarks)
2. Creates the Genie Agent, or updates it in place if it already exists
3. Asks a test question through the Genie Conversation API

### Example questions
- Did any app deployments fail, and why?
- Which app features are used the most, and are any returning errors?
- Can the app reach the Hire Right agent endpoint, or are calls failing?
- Who is using the app and when is it busiest?

### Prerequisites
- `13_app_log_analytics_views.ipynb` has run (views exist)
- A SQL warehouse ID (`warehouse_id` widget)


## Setup

In [ ]:
%pip install databricks-sdk python-dotenv -q

In [ ]:
dbutils.widgets.text("app_log_catalog",      "bldemos",                    "UC Catalog (app telemetry)")
dbutils.widgets.text("app_log_schema",       "hrd_2030_app_observability", "UC Schema (app telemetry)")
dbutils.widgets.text("app_log_table_prefix", "hrd_app",                    "Telemetry Table Prefix")
dbutils.widgets.text("warehouse_id",         "",                           "SQL Warehouse ID")

In [ ]:
import json, os, time, requests

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root  = "/Workspace" + notebook_path.rsplit("/notebooks", 1)[0]
try:
    from dotenv import load_dotenv
    load_dotenv(f"{project_root}/.env")
except ImportError:
    pass

# Widget takes priority (job passes it); fall back to .env for interactive runs
app_log_catalog      = dbutils.widgets.get("app_log_catalog")      or os.getenv("APP_LOG_CATALOG", "bldemos")
app_log_schema       = dbutils.widgets.get("app_log_schema")       or os.getenv("APP_LOG_SCHEMA", "hrd_2030_app_observability")
app_log_table_prefix = dbutils.widgets.get("app_log_table_prefix") or os.getenv("APP_LOG_TABLE_PREFIX", "hrd_app")
warehouse_id         = dbutils.widgets.get("warehouse_id")         or os.getenv("DATABRICKS_WAREHOUSE_ID", "")
if not warehouse_id:
    raise ValueError("warehouse_id is required for the Genie Agent.")

ctx     = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host    = ctx.apiUrl().get().rstrip("/")
headers = {"Authorization": f"Bearer {ctx.apiToken().get()}", "Content-Type": "application/json"}

print(f"Views        : {app_log_catalog}.{app_log_schema}.v_app_*")
print(f"Raw logs     : {app_log_catalog}.{app_log_schema}.{app_log_table_prefix}_otel_logs")
print(f"Warehouse ID : {warehouse_id}")

## Load Definition & Create / Update the Genie Agent

In [ ]:
genie_json_path = f"/Workspace{os.path.dirname(notebook_path)}/14_app_log_genie_space.json"
with open(genie_json_path) as f:
    genie_meta = json.load(f)

serialized_space = (
    json.dumps(genie_meta["serialized_space"])
    .replace("__CATALOG__", app_log_catalog)
    .replace("__SCHEMA__", app_log_schema)
    .replace("__PREFIX__", app_log_table_prefix)
)
title = genie_meta["title"]

payload = {
    "warehouse_id":     warehouse_id,
    "title":            title,
    "description":      genie_meta["description"],
    "serialized_space": serialized_space,
}

# Reuse an existing agent with the same title so re-runs update in place
genie_space_id = ""
_resp = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers, timeout=30)
_resp.raise_for_status()
for _s in _resp.json().get("spaces", []):
    if _s.get("title") == title:
        genie_space_id = _s["space_id"]
        break

if genie_space_id:
    resp = requests.patch(f"{host}/api/2.0/genie/spaces/{genie_space_id}", headers=headers, json=payload, timeout=60)
    action = "Updated"
else:
    resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload, timeout=60)
    action = "Created"
if resp.status_code not in (200, 201):
    raise RuntimeError(f"Genie API error {resp.status_code}: {resp.text[:1000]}")
genie_space_id = genie_space_id or resp.json()["space_id"]

print(f"✓ {action} Genie Agent: {title}")
print(f"  Space ID : {genie_space_id}")
print(f"  URL      : {host}/genie/rooms/{genie_space_id}")

## Test the Genie Agent

In [ ]:
test_question = "Did any app deployments fail, and why?"
print(f"Q: {test_question}\n")

start = requests.post(
    f"{host}/api/2.0/genie/spaces/{genie_space_id}/start-conversation",
    headers=headers, json={"content": test_question}, timeout=60,
)
start.raise_for_status()
conv_id = start.json()["conversation_id"]
msg_id  = start.json()["message_id"]
poll_url = f"{host}/api/2.0/genie/spaces/{genie_space_id}/conversations/{conv_id}/messages/{msg_id}"

msg = {}
for _ in range(60):
    msg = requests.get(poll_url, headers=headers, timeout=30).json()
    if msg.get("status") in ("COMPLETED", "FAILED", "CANCELLED"):
        break
    time.sleep(3)

print(f"Status: {msg.get('status')}")
for att in msg.get("attachments", []):
    if att.get("text"):
        print(att["text"].get("content", ""))
    if att.get("query"):
        print("\nSQL:\n" + att["query"].get("query", ""))
        _res = requests.get(
            f"{poll_url}/attachments/{att['attachment_id']}/query-result", headers=headers, timeout=30
        ).json()
        _sr = _res.get("statement_response", {})
        _cols = [c["name"] for c in _sr.get("manifest", {}).get("schema", {}).get("columns", [])]
        for _row in _sr.get("result", {}).get("data_array", [])[:10]:
            print(dict(zip(_cols, _row)))

## Summary

The **Hire Right App — Log Observability Genie** is live over the five `v_app_*` views plus the raw
`<prefix>_otel_logs` table. Because the views sit directly on the UC telemetry table, every new app log
line is queryable in Genie within seconds.

In [ ]:
try:
    dbutils.jobs.taskValues.set(key="app_log_genie_space_id", value=genie_space_id)
except Exception:
    pass
dbutils.notebook.exit(genie_space_id)